# Iports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import gzip
import sklearn
import matplotlib

import numpy as np
import pandas as pd

from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

import os

from sklearn.decomposition import PCA, FastICA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import LocallyLinearEmbedding as LLE

from sklearn.base import BaseEstimator, TransformerMixin
import time

from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.metrics import silhouette_score, davies_bouldin_score
import urllib.request

import json
import time
from joblib import Parallel, delayed
from sklearn.base import clone
from filelock import FileLock
from datetime import datetime
import umap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.manifold import trustworthiness
from tslearn.datasets import UCR_UEA_datasets
from sklearn.utils import check_random_state

In [4]:
print(torch.__version__)       # wersja PyTorch
print(torch.version.cuda)      # wersja CUDA, na którą PyTorch jest skompilowany
print(torch.backends.cudnn.version())  # wersja cuDNN
print(torch.cuda.is_available())       # czy PyTorch widzi GPU
print(torch.cuda.get_device_name(0))   # nazwa GPU (jeśli dostępna)
# print("Python:", sys.version) 
print("NumPy:", np.__version__) 
print("Pandas:", pd.__version__) 
print("Scikit-learn:", sklearn.__version__) 
# print("TensorFlow:", tf.__version__) 
print("Matplotlib:", matplotlib.__version__) 
print("Seaborn:", sns.__version__) 
print("UMAP-learn:", umap.__version__)

2.10.0+cu128
12.8
91002
True
NVIDIA GeForce RTX 5060 Ti
NumPy: 2.3.5
Pandas: 3.0.0
Scikit-learn: 1.8.0
Matplotlib: 3.10.8
Seaborn: 0.13.2
UMAP-learn: 0.5.11


# Funkcje

In [2]:

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)

        if y_proba.shape[1] == 2:
            # binary
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            # multi-class
            auc = roc_auc_score(y_test, y_proba, multi_class="ovr")
    else:
        auc = None

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted"),
        "Recall": recall_score(y_test, y_pred, average="weighted"),
        "F1": f1_score(y_test, y_pred, average="weighted"),
        "AUC": auc
    }

class ImageScaler:
    def fit_transform(self, x):
        return x.astype(np.float32) / 255.0
    
    def transform(self, x):
        return x.astype(np.float32) / 255.0
    
def log_result(file, reducer, metrics):
    
    lock = FileLock(file + ".lock")
    record = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "reducer": reducer,
        # "classifier": classifier,
        **metrics
    }

    with lock:
        with open(file, "a") as f:
            f.write(json.dumps(record) + "\n")

def evaluate_roc_auc(model, X_test, y_test, title="ROC Curve"):
    y_score = model.predict_proba(X_test)[:, 1]
    return plot_roc_auc(y_test, y_score, title)


def plot_roc_auc(y_true, y_score, title="ROC Curve"):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

    return roc_auc


def plot_and_save_roc(y_true_bin, y_score, class_label, model_name, save_dir):
    fpr, tpr, _ = roc_curve(y_true_bin, y_score)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC – klasa {class_label} ({model_name})")
    plt.legend()
    plt.grid(True)

    filename = f"{save_dir}/roc_{model_name}_class_{class_label}.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()

    return roc_auc


def evaluate_multiclass_roc_auc(
    model,
    X_test,
    y_test,
    model_name="model",
    save_dir="roc_plots"
):
    """
    Oblicza ROC i AUC dla wszystkich klas w trybie One-vs-Rest,
    rysuje i zapisuje wykresy do plików.
    Działa zarówno dla modeli z predict_proba jak i SVM z decision_function.
    Zwraca DataFrame z AUC dla każdej klasy.
    """
    
    os.makedirs(save_dir, exist_ok=True)
    
    # Obsługa predict_proba lub decision_function
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
        # dla multi-class decision_function może zwracać shape (n_samples, n_classes)
        if len(y_score.shape) == 1:  # binarna klasyfikacja
            y_score = np.vstack([1 - y_score, y_score]).T
    else:
        raise ValueError("Model musi mieć predict_proba lub decision_function")
    
    y_score = model.predict_proba(X_test)
    classes = np.unique(y_test)

    y_test_bin = label_binarize(y_test, classes=classes)

    auc_results = []

    for i, cls in enumerate(classes):
        auc_cls = plot_and_save_roc(
            y_test_bin[:, i],
            y_score[:, i],
            class_label=cls,
            model_name=model_name,
            save_dir=save_dir
        )

        auc_results.append({
            "model": model_name,
            "class": cls,
            "auc": auc_cls
        })

    return pd.DataFrame(auc_results)


def reduction_results_to_df(method_name, reduction_results):
    """
    Konwertuje wyniki redukcji do DataFrame.
    
    method_name: nazwa metody redukcji
    reduction_results: wynik z full_pipeline_cv
    
    Zwraca DataFrame:
    ['Method','Silhouette_mean','Silhouette_std',
     'DBI_mean','DBI_std',
     'Reconstruction_mean','Reconstruction_std',
     'Train_time_mean','Train_time_std']
    """

    row = {
        "Method": method_name,
        "Silhouette_mean": reduction_results["Silhouette"]["mean"],
        "Silhouette_std": reduction_results["Silhouette"]["std"],
        "DBI_mean": reduction_results["DBI"]["mean"],
        "DBI_std": reduction_results["DBI"]["std"],
        "Reconstruction_mean": reduction_results["Reconstruction_Error"]["mean"],
        "Reconstruction_std": reduction_results["Reconstruction_Error"]["std"],
        "Train_time_mean": reduction_results["Train_time"]["mean"],
        "Train_time_std": reduction_results["Train_time"]["std"]
    }

    return pd.DataFrame([row])

def classification_results_to_df(method_name, classification_results):
    """
    Konwertuje wyniki klasyfikacji do DataFrame.

    classification_results: wynik z full_pipeline_cv
    
    Kolumny:
    ['Method','Classifier','Accuracy_mean','Accuracy_std',
     'F1_mean','F1_std','Train_time_mean','Train_time_std']
    """

    rows = []

    for clf_name, metrics in classification_results.items():

        row = {
            "Method": method_name,
            "Classifier": clf_name
        }

        # dodajemy wszystkie metryki dynamicznie
        for key, values in metrics.items():
            row[f"{key}_mean"] = values["mean"]
            row[f"{key}_std"] = values["std"]

        rows.append(row)

    return pd.DataFrame(rows)


def build_pipeline(reducer, classifier):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("reducer", reducer),
        ("classifier", classifier)
    ])


def run_experiment(X, y, reducer, classifier, n_splits=5, n_repeats=3):
    
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=765243
    )
    
    acc_scores = []
    f1_scores = []
    train_times = []
    
    for train_idx, test_idx in cv.split(X, y):
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        pipeline = build_pipeline(reducer, classifier)
        
        start = time.time()
        pipeline.fit(X_train, y_train)
        train_time = time.time() - start
        
        y_pred = pipeline.predict(X_test)
        
        acc_scores.append(accuracy_score(y_test, y_pred))
        f1_scores.append(f1_score(y_test, y_pred, average="weighted"))
        train_times.append(train_time)
    
    return {
        "Accuracy_mean": np.mean(acc_scores),
        "Accuracy_std": np.std(acc_scores),
        "F1_mean": np.mean(f1_scores),
        "F1_std": np.std(f1_scores),
        "Train_time_mean": np.mean(train_times)
    }

def trust_sample(X, X_latent, n_neighbors=5, sample_size=2000, random_state=42):
    rng = check_random_state(random_state)
    idx = rng.choice(X.shape[0], min(sample_size, X.shape[0]), replace=False)
    return trustworthiness(X[idx], X_latent[idx], n_neighbors=n_neighbors)

def process_fold(train_idx, test_idx, X, y, reducer_getter, classifiers, print_train_plot=False, reducer_name = '', log_name = '', scaler_getter = None):

    X_train = np.asarray(X[train_idx])
    X_test  = np.asarray(X[test_idx])
    y_train = y[train_idx]
    y_test  = y[test_idx]

    if scaler_getter == None:
        scaler =  StandardScaler()
    else:
        scaler = scaler_getter()
        
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    start_red = time.time()
    reducer = reducer_getter()
    print(X_train_scaled.dtype)
    reducer.fit(X_train_scaled, y_train)
    train_time_red = time.time() - start_red

    X_train_latent = reducer.transform(X_train_scaled)

    sil_score = silhouette_score(X_train_latent, y_train)
    dbi_score = davies_bouldin_score(X_train_latent, y_train)
    trust = trust_sample(X_train, X_train_latent, n_neighbors=5)

    recon_error = None
    if hasattr(reducer, "reconstruct"):
        with torch.no_grad():
            recon = reducer.reconstruct(X_train_latent, X_train_scaled)
            X_train_scaled_reshaped = reducer.reshape_X(X_train_scaled)
            recon_error = float(np.mean((recon - X_train_scaled_reshaped)**2))
            print(recon_error)
    if print_train_plot and hasattr(reducer, "train_losses_"):
        plt.figure()
        plt.plot(reducer.train_losses_, label="Train")
        plt.plot(reducer.val_losses_, label="Validation")
        plt.legend()
        plt.title("Autoencoder Training")
        plt.show()

    reduction_metrics = {
        "Silhouette": sil_score,
        "DBI": dbi_score,
        "Reconstruction_Error": recon_error,
        "Train_time": train_time_red,
        "Trustworthiness": trust,
    }

    classification_metrics = {}

    for name, clf_getter in classifiers.items():
        clf = clf_getter()
        clf.fit(X_train_latent, y_train)

        start_clf = time.time()
        X_test_latent = reducer.transform(X_test_scaled)
        y_pred = clf.predict(X_test_latent)
        pred_time = time.time() - start_clf

        classification_metrics[name] = {
            **evaluate_model(clf, X_test_latent, y_test),
            "Prediction_time": pred_time
        }
    
    log_result(
            f'{log_name}.json',
            reducer=reducer_name,
            metrics={'reduction_metrics': reduction_metrics, 'classification_metrics': classification_metrics}
        )

    return reduction_metrics, classification_metrics


def full_pipeline_cv(X, y, reducer_getter, classifiers, n_splits=5, n_repeats=3, print_train_plot = False, reducer_name = '', log_name = '', scaler_getter = None):
    """
    Jeden pipeline dla wszystkich metod i wszystkich zbiorów:
    - redukcja wymiarów
    - metryki jakości redukcji
    - klasyfikacja
    - czas uczenia
    """
        
    # przygotowanie CV
    cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=765243)
    if reducer_name in ["AE", "VAE"]:
        # n_jobs_fold = 4
        results = []
        for train_idx, test_idx in cv.split(X, y):
            results.append(process_fold(
                train_idx,
                test_idx,
                X,
                y,
                reducer_getter,
                classifiers,
                print_train_plot,
                reducer_name,
                log_name,
                scaler_getter
            ))
        
    else:
        n_jobs_fold = 3

        results = Parallel(n_jobs=n_jobs_fold)(
            delayed(process_fold)(
                train_idx,
                test_idx,
                X,
                y,
                reducer_getter,
                classifiers,
                print_train_plot,
                reducer_name,
                log_name,
                scaler_getter
            )
            for train_idx, test_idx in cv.split(X, y)
        )
    
    reduction_metrics = []
    classification_metrics = {name: [] for name in classifiers.keys()}

    for red_res, clf_res in results:

        reduction_metrics.append(red_res)

        for name in clf_res:
            classification_metrics[name].append(clf_res[name])

    
    reduction_summary = {}

    keys = reduction_metrics[0].keys()  # wszystkie metryki obecne w słownikach
    reduction_summary = {
        k: {
                "mean": np.mean([m[k] for m in reduction_metrics if m[k] is not None]),
                "std": np.std([m[k] for m in reduction_metrics if m[k] is not None])
            }
            for k in keys
    }
    
    classification_summary = {}
    for clf_name, results in classification_metrics.items():
        keys = results[0].keys() 
        classification_summary[clf_name] = {k: {"mean": np.mean([r[k] for r in results]),
                                                "std": np.std([r[k] for r in results])}
                                            for k in keys}
    
    return reduction_summary, classification_summary


def full_pipeline(X, y, reducers, classifiers, n_splits = 5, n_repeats=3, name = 'wynik', print_train_plot = False, scaler_getter= None):
    reduction_tables = []
    classification_tables = []

    for reducer_name, reducer in reducers.items():
        print(f'full_pipeline: {reducer_name}')
        reduction_results, classification_results = full_pipeline_cv(
                X,
                y,
                reducer,
                classifiers,
                reducer_name = reducer_name,
                print_train_plot = print_train_plot,
                log_name=name,
                n_splits=n_splits,
                n_repeats=n_repeats,
                scaler_getter = scaler_getter
            )

        reduction_tables.append(
                reduction_results_to_df(reducer_name, reduction_results)
            )

        classification_tables.append(
                classification_results_to_df(reducer_name, classification_results)
            )
        
        reduction_df = pd.concat(reduction_tables, ignore_index=True)
        classification_df = pd.concat(classification_tables, ignore_index=True)

        # reduction_df.to_latex(f"{name}_reduction_results.tex", index=False)
        # classification_df.to_latex(f"{name}_classification_results.tex", index=False)
               
        # zapis do pliku LaTeX po każdej iteracji (dopisywanie)
        with open(f"{name}_reduction_results.tex", 'a') as f:
            f.write(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            f.write("\n")
            f.write(reduction_df.to_latex(index=False))
            f.write("\n\n")  # przerwa między tabelami

        with open(f"{name}_classification_results.tex", 'a') as f:
            f.write(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            f.write("\n")
            f.write(classification_df.to_latex(index=False))
            f.write("\n\n")

        # opcjonalnie druk w konsoli
        print(reduction_df)
        print(classification_df)



    # print(reduction_df)
    # classification_df


# Definicje auto-endkoderow

In [3]:
    
class BaseAutoencoder(BaseEstimator, TransformerMixin):

    def __init__(self, latent_dim=2, epochs=100, lr=1e-3, batch_size=32,
                 val_split=0.1):
        self.latent_dim = latent_dim
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.val_split = val_split

    def build_model(self, input_dim):
        """
        Ta metoda MUSI być nadpisana w klasie pochodnej 
        """
        raise NotImplementedError()
    
    def reshape_X(self, X):
        print('no reshape_X implemented')
        return X

    def fit(self, X, y=None):
        X = self.reshape_X(X)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if self.val_split == None:
            X_train = X
        else:
            X_train, X_val = train_test_split(
                X,
                test_size=self.val_split,
                random_state=765243
            )

        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        train_dataset = torch.utils.data.TensorDataset(X_train_tensor)

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

        if self.val_split != None:
            X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)

        input_dim = X.shape[1]

        self.encoder, self.decoder = self.build_model(input_dim)

        self.encoder.to(device)
        self.decoder.to(device)

        params = list(self.encoder.parameters()) + list(self.decoder.parameters())

        optimizer = optim.Adam(params, lr=self.lr)
        criterion = nn.MSELoss()

        self.train_losses_ = []
        self.val_losses_ = []

        start = time.time()

        for epoch in range(self.epochs):
            print(f"epoch: {epoch}")

            self.encoder.train()
            self.decoder.train()

            train_epoch_loss = 0

            for batch in train_loader:

                x_batch = batch[0].to(device)

                optimizer.zero_grad()

                z = self.encoder(x_batch)
                recon = self.decoder(z)

                loss = criterion(recon, x_batch)

                loss.backward()
                optimizer.step()

                train_epoch_loss += loss.item() * x_batch.size(0)

            train_epoch_loss /= len(train_loader.dataset)
            self.train_losses_.append(train_epoch_loss)

            # ===== VALIDATION =====

            if self.val_split:
                self.encoder.eval()
                self.decoder.eval()

                with torch.no_grad():

                    z = self.encoder(X_val_tensor)
                    recon = self.decoder(z)

                    val_loss = criterion(recon, X_val_tensor).item()

                self.val_losses_.append(val_loss)

        self.train_time_ = time.time() - start

        return self

    def transform(self, X):
        X = self.reshape_X(X)
        device = next(self.encoder.parameters()).device

        with torch.no_grad():
            X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
            z = self.encoder(X_tensor)

        return z.cpu().numpy()

    def reconstruct(self, X, _):

        device = next(self.encoder.parameters()).device

        with torch.no_grad():

            X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

            recon = self.decoder(X_tensor)

        return recon.cpu().numpy()
    
class BaseVAE(BaseEstimator, TransformerMixin):

    def __init__(self, latent_dim=2, epochs=100, lr=1e-3, batch_size=32,
                 val_split=0.1):
        self.latent_dim = latent_dim
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.val_split = val_split

    def build_model(self, input_dim):
        raise NotImplementedError()
    
    def reshape_X(self, X):
        return X
    
    def reparameterize(self, mu, logvar):

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mu + eps * std

    def loss_function(self, recon_x, x, mu, logvar):

        recon_loss = nn.MSELoss()(recon_x, x)

        kl = -0.5 * torch.sum(
            1 + logvar - mu.pow(2) - logvar.exp()
        )

        return recon_loss + kl

    def fit(self, X, y=None):
        X = self.reshape_X(X)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if self.val_split == None:
            X_train = X
        else:
            X_train, X_val = train_test_split(
                X,
                test_size=self.val_split,
                random_state=765243
            )

        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)

        train_dataset = torch.utils.data.TensorDataset(X_train_tensor)

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )
        if self.val_split: 
            X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)

        input_dim = X.shape[1]

        self.encoder, self.mu_layer, self.logvar_layer, self.decoder = self.build_model(input_dim)

        self.encoder.to(device)
        self.mu_layer.to(device)
        self.logvar_layer.to(device)
        self.decoder.to(device)

        params = (
            list(self.encoder.parameters()) +
            list(self.mu_layer.parameters()) +
            list(self.logvar_layer.parameters()) +
            list(self.decoder.parameters())
        )

        optimizer = optim.Adam(params, lr=self.lr)

        self.train_losses_ = []
        self.val_losses_ = []

        start = time.time()

        for epoch in range(self.epochs):
            print(f"epoch: {epoch}")
            self.encoder.train()

            train_epoch_loss = 0

            for batch in train_loader:

                x = batch[0].to(device)

                optimizer.zero_grad()

                h = self.encoder(x)

                mu = self.mu_layer(h)
                logvar = self.logvar_layer(h)

                z = self.reparameterize(mu, logvar)

                recon = self.decoder(z)

                loss = self.loss_function(recon, x, mu, logvar)

                loss.backward()
                optimizer.step()

                train_epoch_loss += loss.item() * x.size(0)

            train_epoch_loss /= len(train_loader.dataset)
            self.train_losses_.append(train_epoch_loss)

            # ===== VALIDATION =====

            if self.val_split:
                self.encoder.eval()

                with torch.no_grad():

                    h = self.encoder(X_val_tensor)

                    mu = self.mu_layer(h)
                    logvar = self.logvar_layer(h)

                    z = self.reparameterize(mu, logvar)

                    recon = self.decoder(z)

                    val_loss = self.loss_function(recon, X_val_tensor, mu, logvar).item()

                self.val_losses_.append(val_loss)

        self.train_time_ = time.time() - start

        return self
    
    def transform(self, X):
        
        X = self.reshape_X(X)
        device = next(self.encoder.parameters()).device

        with torch.no_grad():

            X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
            h = self.encoder(X_tensor)
            mu = self.mu_layer(h)
            # logvar = self.logvar_layer(h)
            # z = self.reparameterize(mu, logvar)

        return mu.cpu().numpy()

    def reconstruct(self, latent_X, X):
        X = self.reshape_X(X)

        device = next(self.encoder.parameters()).device

        with torch.no_grad():

            X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
            mu = torch.tensor(latent_X, dtype=torch.float32).to(device)
            h = self.encoder(X_tensor)
            logvar = self.logvar_layer(h)
            z = self.reparameterize(mu, logvar)
            recon = self.decoder(z)

        return recon.cpu().numpy()
    

    
class WineVAE(BaseVAE):

    def build_model(self, input_dim):

        encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        mu_layer = nn.Linear(32, self.latent_dim)
        logvar_layer = nn.Linear(32, self.latent_dim)

        decoder = nn.Sequential(
            nn.Linear(self.latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

        return encoder, mu_layer, logvar_layer, decoder
    
class WineAutoencoder(BaseAutoencoder):

    def build_model(self, input_dim):

        encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, self.latent_dim)
        )

        decoder = nn.Sequential(
            nn.Linear(self.latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

        return encoder, decoder

class MNISTAutoencoder(BaseAutoencoder):

    def reshape_X(self, X):
        return  X.reshape(-1,1,28,28)

    def build_model(self, input_dim):

        encoder = nn.Sequential(
            nn.Conv2d(1, 4, 3, stride=2, padding=1),  # 28x28 -> 14x14
            nn.ReLU(),
            nn.Conv2d(4, 8, 3, stride=2, padding=1), # 14x14 -> 7x7
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(8*7*7, self.latent_dim)
        )
        
        # Decoder
        decoder = nn.Sequential(
            nn.Linear(self.latent_dim, 8*7*7),
            nn.ReLU(),
            nn.Unflatten(1, (8,7,7)),
            nn.ConvTranspose2d(8, 4, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(4, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

        return encoder, decoder
        
    
class MNISTVAE(BaseVAE):

    def reshape_X(self, X):
        return  X.reshape(-1,1,28,28)

    def build_model(self, input_shape):
        # ---------------- Encoder ----------------
        encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1), # 28x28 -> 14x14
            nn.ReLU(),
            nn.Conv2d(8, 16, 3, stride=2, padding=1), # 14x14 -> 7x7
            nn.ReLU(),
            nn.Flatten()
        )
        # latent_dim to np. 16
        flatten_dim = 16*7*7
        mu_layer = nn.Linear(flatten_dim, self.latent_dim)
        logvar_layer = nn.Linear(flatten_dim, self.latent_dim)
        
        # ---------------- Decoder ----------------
        decoder = nn.Sequential(
            nn.Linear(self.latent_dim, flatten_dim),
            nn.ReLU(),
            nn.Unflatten(1, (16,7,7)),
            nn.ConvTranspose2d(16,8,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(8, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )
        
        return encoder, mu_layer, logvar_layer, decoder

class ECGAutoencoder(BaseAutoencoder):

    def reshape_X(self, X):
        return X  # już jest (n_samples, length)

    def build_model(self, input_dim):

        encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, self.latent_dim)
        )

        decoder = nn.Sequential(
            nn.Linear(self.latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

        return encoder, decoder
    
class ECGConvAutoencoder(BaseAutoencoder):

    def reshape_X(self, X):
        return X.reshape(-1, 1, 96)  # (batch, channel, length)

    def build_model(self, input_dim):

        encoder = nn.Sequential(
            nn.Conv1d(1, 6, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv1d(6, 12, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear((96 // 4) * 12, self.latent_dim)
        )

        decoder = nn.Sequential(
            nn.Linear(self.latent_dim, (96 // 4) * 12),
            nn.ReLU(),
            nn.Unflatten(1, (12, 96 // 4)),
            nn.ConvTranspose1d(12, 6, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(6, 1, kernel_size=3, stride=2, padding=1, output_padding=1)
        )

        return encoder, decoder

class BasicMotionsAE(BaseAutoencoder):

    def reshape_X(self, X):
        return X  # flatten

    def build_model(self, input_dim):

        encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, self.latent_dim)
        )

        decoder = nn.Sequential(
            nn.Linear(self.latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

        return encoder, decoder
    
class BasicMotionsConvAE(BaseAutoencoder):

    def reshape_X(self, X):
        return X.reshape(-1, 6, 100)

    def build_model(self, input_dim):

        n_channels = 6  # np. 6
        length = 100      # np. 100

        encoder = nn.Sequential(
            nn.Conv1d(n_channels, 8, kernel_size=3, stride=2, padding=1),   # -> length/2
            nn.ReLU(),
            nn.Conv1d(8, 16, kernel_size=3, stride=2, padding=1),           # -> length/4
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear((length // 4) * 16, self.latent_dim)
        )

        decoder = nn.Sequential(
            nn.Linear(self.latent_dim, (length // 4) * 16),
            nn.ReLU(),
            nn.Unflatten(1, (16, length // 4)),
            nn.ConvTranspose1d(16, 8, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(8, n_channels, kernel_size=3, stride=2, padding=1, output_padding=1)
        )

        return encoder, decoder
    


# 

In [ ]:
data = load_wine()

X = data.data
y = data.target

reducers = {    
    "PCA": lambda: PCA(n_components=4),
    "ICA": lambda: FastICA(
        n_components=4,
        random_state=765243
    ),
    "LLE": lambda: LocallyLinearEmbedding(
        n_components=4,
        n_neighbors=10
    ),
    "LDA": lambda: LDA(n_components=2),
    "AE": lambda: WineAutoencoder(
        latent_dim=4,
        epochs=50,
        val_split=None
    ),
    "VAE": lambda: WineVAE(
        latent_dim=4,
        epochs=50,
        val_split=None
    )
}

classifiers = {
    "SVM": lambda: SVC(kernel="rbf", probability=True, random_state=765243),
    "MLP": lambda: MLPClassifier(hidden_layer_sizes=(64,16), activation="relu", max_iter=100, random_state=765243),
    "Decision Tree": lambda: DecisionTreeClassifier(max_depth=10, random_state=765243)
}

full_pipeline(X, y, reducers, classifiers, name='Wine_4')


full_pipeline: PCA
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64


c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.389794        0.010586  1.012354  0.030799   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.001125        0.000158  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.971746      0.025441        0.973069   
1    PCA            MLP       0.958677      0.026968        0.961605   
2    PCA  Decision Tree       0.919312      0.027304        0.925950   

   Precision_std  Recall_mean  Recall_std   F1_mean    F1_std  AUC_mean  \
0       0.024876     0.971746    0.025441  0.971591  0.025627  0.997112   
1       0.025284     0.958677    0.026968  0.958436  0.027048  0.997102   
2       0.025818     0.919312    0.027304  0.918969  0.027513  0.955372   

    AUC_std  Prediction_time_mean  Prediction_time_std  
0  0.004929              0.000559             0.000082  
1  0.0038

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.389794        0.010586  1.012354  0.030799   
1    ICA         0.299748        0.007581  1.362140  0.029502   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.001125        0.000158  
1                  NaN                 NaN         0.004336        0.002624  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.971746      0.025441        0.973069   
1    PCA            MLP       0.958677      0.026968        0.961605   
2    PCA  Decision Tree       0.919312      0.027304        0.925950   
3    ICA            SVM       0.969841      0.026443        0.971202   
4    ICA            MLP       0.966032      0.029738        0.969591   
5    ICA  Decision Tree       0.951058      0.035337        0.954694   

   Precision_std  Recall_mean  Recall_std   F1_mean    F1_std  AU

...

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoch: 48
epoch: 49
1.0021507005999875
float64
epoch: 0


c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoch: 48
epoch: 49
1.0011078852475448
  Method  Silhouette_mean  Silhouette_std   DBI_mean   DBI_std  \
0    PCA         0.389794        0.010586   1.012354  0.030799   
1    ICA         0.299748        0.007581   1.362140  0.029502   
2    LLE         0.248768        0.055426   1.653267  0.311727   
3    LDA         0.667396        0.009457   0.442460  0.011387   
4     AE         0.382511        0.042226   1.001684  0.148068   
5    VAE        -0.027800        0.007260  18.175223  8.079467   

   Reconstruction_mean  Reconstructio

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

reducers = {    
    "PCA": lambda: PCA(n_components=4),
    "ICA": lambda: FastICA(
        n_components=4,
        random_state=765243
    ),
    "LLE": lambda: LocallyLinearEmbedding(
        n_components=4,
        n_neighbors=10
    ),
    "LDA": lambda: LDA(n_components=1),  
    "AE": lambda: WineAutoencoder(
        latent_dim=4,
        epochs=50,
        val_split=None
    ),
    "VAE": lambda: WineVAE(
        latent_dim=4,
        epochs=50,
        val_split=None
    )
}

classifiers = {
    "SVM": lambda: SVC(kernel="rbf", probability=True, random_state=765243),
    "MLP": lambda: MLPClassifier(
        hidden_layer_sizes=(64, 16),
        activation="relu",
        max_iter=100,
        random_state=765243
    ),
    "Decision Tree": lambda: DecisionTreeClassifier(
        max_depth=10,
        random_state=765243
    ),
    "kNN": lambda: KNeighborsClassifier(n_neighbors=10) 
}

full_pipeline(X, y, reducers, classifiers, name="BreastCancer_4")

full_pipeline: PCA


c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.359714        0.006519  1.174706  0.016667   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.001287        0.001183  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.950800      0.020993        0.951173   
1    PCA            MLP       0.953108      0.020563        0.953911   
2    PCA  Decision Tree       0.916814      0.019964        0.918779   

   Precision_std  Recall_mean  Recall_std   F1_mean    F1_std  AUC_mean  \
0       0.020897     0.950800    0.020993  0.950635  0.021121  0.991475   
1       0.020337     0.953108    0.020563  0.953201  0.020558  0.989468   
2       0.019545     0.916814    0.019964  0.916876  0.019954  0.912993   

    AUC_std  Prediction_time_mean  Prediction_time_std  
0  0.006059              0.001363             0.000220  
1  0.0078

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.359714        0.006519  1.174706  0.016667   
1    ICA         0.226683        0.003614  1.912422  0.019633   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.001287        0.001183  
1                  NaN                 NaN         0.008407        0.002570  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.950800      0.020993        0.951173   
1    PCA            MLP       0.953108      0.020563        0.953911   
2    PCA  Decision Tree       0.916814      0.019964        0.918779   
3    ICA            SVM       0.954898      0.019482        0.955054   
4    ICA            MLP       0.951964      0.023365        0.953052   
5    ICA  Decision Tree       0.919769      0.018840        0.920598   

   Precision_std  Recall_mean  Recall_std   F1_mean    F1_std  AU

...

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


float64
epoch: 0
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoch: 48
epoch: 49
1.0002449349698452


c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


float64
epoch: 0
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoch: 48
epoch: 49
0.9995898057862821
  Method  Silhouette_mean  Silhouette_std   DBI_mean   DBI_std  \
0    PCA         0.359714        0.006519   1.174706  0.016667   
1    ICA         0.226683        0.003614   1.912422  0.019633   
2    LLE         0.232359        0.047188   1.826975  0.178931   
3    LDA         0.681479        0.009163   0.433324  0.013437   
4     AE         0.337449        0.018243   1.236622  0.088008   
5    VAE         0.042174        0.020049  11.745529  4.746159   

   Reconstruction_me

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
ucr = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ucr.load_dataset("ECG200")

# Połączenie train + test (do EDA)
X = np.concatenate((X_train, X_test))
X = X.reshape(X.shape[0], -1)
y = np.concatenate((y_train, y_test))

n_components =8
LDA_n_components = min(n_components, len(np.unique(y))- 1)

reducers = {    
    "PCA": lambda: PCA(n_components=n_components),
    "ICA": lambda: FastICA(
        n_components=n_components,
        random_state=765243
    ),
    "LLE": lambda: LocallyLinearEmbedding(
        n_components=n_components,
        n_neighbors=n_components
    ),
    "LDA": lambda: LDA(n_components=LDA_n_components),
    "AE": lambda: ECGAutoencoder(
        latent_dim=n_components,
        epochs=50,
        val_split=None
    ),
    "ConvAE": lambda: ECGConvAutoencoder(
        latent_dim=n_components,
        epochs=50,
        val_split=None
    )
}

classifiers = {
    "SVM": lambda: SVC(kernel="rbf", probability=True, random_state=765243),
    "MLP": lambda: MLPClassifier(hidden_layer_sizes=(64,16), activation="relu", max_iter=100, random_state=765243),
    "Decision Tree": lambda: DecisionTreeClassifier(max_depth=10, random_state=765243),
    "kNN": lambda: KNeighborsClassifier(n_neighbors=10) 
}


full_pipeline(X, y, reducers, classifiers, name='ECG_8')

full_pipeline: PCA
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64
float64


c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.230126        0.012352  2.396739  0.105694   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.025585        0.001609  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.876667      0.037044        0.876423   
1    PCA            MLP       0.875000      0.051640        0.876948   
2    PCA  Decision Tree       0.800000      0.054006        0.804405   
3    PCA            kNN       0.846667      0.046428        0.847687   

   Precision_std  Recall_mean  Recall_std   F1_mean    F1_std  AUC_mean  \
0       0.037995     0.876667    0.037044  0.874633  0.038750  0.936250   
1       0.051975     0.875000    0.051640  0.872920  0.054260  0.925248   
2       0.051047     0.800000    0.054006  0.797283  0.051257  0.765161   
3       0.047306     0.846667    0.046428  0.845520 

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.230126        0.012352  2.396739  0.105694   
1    ICA         0.154499        0.005669  3.783595  0.092029   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.025585        0.001609  
1                  NaN                 NaN         0.037603        0.005563  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.876667      0.037044        0.876423   
1    PCA            MLP       0.875000      0.051640        0.876948   
2    PCA  Decision Tree       0.800000      0.054006        0.804405   
3    PCA            kNN       0.846667      0.046428        0.847687   
4    ICA            SVM       0.896667      0.035198        0.897096   
5    ICA            MLP       0.880000      0.044907        0.880975   
6    ICA  Decision Tree       0.803333      0.067618        0.8074

...

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.230126        0.012352  2.396739  0.105694   
1    ICA         0.154499        0.005669  3.783595  0.092029   
2    LLE         0.157668        0.013018  3.930044  0.159920   
3    LDA         0.773727        0.019273  0.297602  0.022674   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.025585        0.001609  
1                  NaN                 NaN         0.037603        0.005563  
2                  NaN                 NaN         0.029815        0.000943  
3                  NaN                 NaN         0.027096        0.001861  
   Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0     PCA            SVM       0.876667      0.037044        0.876423   
1     PCA            MLP       0.875000      0.051640        0.876948   
2     PCA  Decision Tree       0.800000      0.054006        0.80

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


float64
epoch: 0
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoch: 48
epoch: 49
0.17823680960767202
float64
epoch: 0
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoc

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


float64
epoch: 0
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoch: 48
epoch: 49
0.17449460415867468
float64
epoch: 0
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoc

In [ ]:
ucr = UCR_UEA_datasets()

X_train, y_train, X_test, y_test = ucr.load_dataset("BasicMotions")

X = np.concatenate((X_train, X_test))
X = X.reshape(X.shape[0], -1)
y = np.concatenate((y_train, y_test))

n_components =12
LDA_n_components = min(n_components, len(np.unique(y))- 1)

reducers = {    
    "PCA": lambda: PCA(n_components=n_components),
    "ICA": lambda: FastICA(
        n_components=n_components,
        random_state=765243
    ),
    "LLE": lambda: LocallyLinearEmbedding(
        n_components=n_components,
        n_neighbors=10
    ),
    "LDA": lambda: LDA(n_components=LDA_n_components), 
    "AE": lambda: BasicMotionsAE(
        latent_dim=n_components,
        epochs=50,
        val_split=None
    ),
    "ConvAE": lambda: BasicMotionsConvAE(
        latent_dim=n_components,
        epochs=50,
        val_split=None
    )
}

classifiers = {
    "SVM": lambda: SVC(kernel="rbf", probability=True, random_state=765243),
    "MLP": lambda: MLPClassifier(
        hidden_layer_sizes=(64,16),
        activation="relu",
        max_iter=100,
        random_state=765243
    ),
    "Decision Tree": lambda: DecisionTreeClassifier(
        max_depth=10,
        random_state=765243
    ),
    "kNN": lambda: KNeighborsClassifier(n_neighbors=10) 
}

full_pipeline(X, y, reducers, classifiers, name="BasicMotions_12")

full_pipeline: PCA


c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA        -0.021482         0.00744  3.228889  0.121547   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.005612        0.000736  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM         0.6500      0.081650        0.646706   
1    PCA            MLP         0.7250      0.093541        0.732143   
2    PCA  Decision Tree         0.7375      0.082916        0.751984   

   Precision_std  Recall_mean  Recall_std   F1_mean    F1_std  AUC_mean  \
0       0.111717       0.6500    0.081650  0.619953  0.093647  0.818056   
1       0.124828       0.7250    0.093541  0.701472  0.104202  0.944792   
2       0.121256       0.7375    0.082916  0.722294  0.092803  0.825000   

    AUC_std  Prediction_time_mean  Prediction_time_std  
0  0.029736              0.000509             0.000086  
1  0.0280

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA        -0.021482        0.007440  3.228889  0.121547   
1    ICA        -0.024077        0.008845  3.462629  0.231200   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.005612        0.000736  
1                  NaN                 NaN         0.024093        0.004625  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.650000      0.081650        0.646706   
1    PCA            MLP       0.725000      0.093541        0.732143   
2    PCA  Decision Tree       0.737500      0.082916        0.751984   
3    ICA            SVM       0.650000      0.087797        0.636548   
4    ICA            MLP       0.675000      0.097361        0.684484   
5    ICA  Decision Tree       0.754167      0.112885        0.781667   

   Precision_std  Recall_mean  Recall_std   F1_mean    F1_std  AU

...

float64
epoch: 0
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoch: 48
epoch: 49
0.21044620051696375
float64
epoch: 0
epoch: 1
epoch: 2
epoch: 3


c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
epoch: 28
epoch: 29
epoch: 30
epoch: 31
epoch: 32
epoch: 33
epoch: 34
epoch: 35
epoch: 36
epoch: 37
epoch: 38
epoch: 39
epoch: 40
epoch: 41
epoch: 42
epoch: 43
epoch: 44
epoch: 45
epoch: 46
epoch: 47
epoch: 48
epoch: 49
0.23640376196247306
  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA        -0.021482        0.007440  3.228889  0.121547   
1    ICA        -0.024077        0.008845  3.462629  0.231200   
2    LLE        -0.043906        0.006788  4.231822  0.402563   
3    LDA         0.672546        0.034110  0.611029  0.084032   
4     AE         0.004910        0.023528  3.283569  0.504970   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.0056

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


   Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0     PCA        -0.021482        0.007440  3.228889  0.121547   
1     ICA        -0.024077        0.008845  3.462629  0.231200   
2     LLE        -0.043906        0.006788  4.231822  0.402563   
3     LDA         0.672546        0.034110  0.611029  0.084032   
4      AE         0.004910        0.023528  3.283569  0.504970   
5  ConvAE         0.020148        0.036568  2.570583  0.353106   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.005612        0.000736  
1                  NaN                 NaN         0.024093        0.004625  
2                  NaN                 NaN         0.042564        0.054706  
3                  NaN                 NaN         0.017165        0.001432  
4             0.232652            0.018299         0.887251        0.798506  
5             0.891901            0.019480         1.890770        1.

In [ ]:
MNIST_URL = "https://s3.amazonaws.com/img-datasets/mnist.npz"
DATA_PATH = "mnist.npz"

# Pobranie pliku, jeśli nie istnieje
if not os.path.exists(DATA_PATH):
    print("Pobieranie MNIST...")
    urllib.request.urlretrieve(MNIST_URL, DATA_PATH)
    print("Pobrano MNIST.")

# Wczytanie danych
with np.load(DATA_PATH) as data:
    X_train = data["x_train"]
    y_train = data["y_train"]
    X_test = data["x_test"]
    y_test = data["y_test"]

X = np.concatenate((X_train, X_test))#[:100]
X = X.reshape(X.shape[0], -1)
y = np.concatenate((y_train, y_test))#[:100]
n_components = 32
LDA_n_components = min(n_components, len(np.unique(y))- 1)
print(LDA_n_components)
reducers = {    
    "PCA": lambda: PCA(n_components=n_components),
    "ICA": lambda: FastICA(
        n_components=n_components,
        random_state=765243
    ),
    # "LLE": lambda: LocallyLinearEmbedding(
    #     n_components=n_components,
    #     n_neighbors=10
    # ),
    # "LDA": lambda: LDA(n_components= LDA_n_components),
    # "UMAP": lambda: umap.UMAP(n_components=n_components),
    "ConvAE": lambda: MNISTAutoencoder(
        latent_dim=n_components,
        epochs=50,
        val_split=None
    ),
    # "ConvAE": lambda: MNISTVAE(
    #     latent_dim=n_components,
    #     epochs=30,
    #     val_split=None
    # )
}

scaler_getter = lambda: ImageScaler()

classifiers = {
    "SVM": lambda: SVC(kernel="rbf", probability=True, random_state=765243),
    "MLP": lambda: MLPClassifier(hidden_layer_sizes=(64,16), activation="relu", max_iter=100, random_state=765243),
    "Decision Tree": lambda: DecisionTreeClassifier(max_depth=10, random_state=765243),
    "kNN": lambda: KNeighborsClassifier(n_neighbors=10) 
}

full_pipeline(X, y, reducers, classifiers, name='Mnist_32', print_train_plot=True, n_splits=5, n_repeats=3, scaler_getter=scaler_getter)


9
full_pipeline: PCA


c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.073573        0.000247  3.273141  0.004376   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.626432        0.093778  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.981162      0.000780        0.981166   
1    PCA            MLP       0.968395      0.001817        0.968435   
2    PCA  Decision Tree       0.793114      0.004722        0.798538   
3    PCA            kNN       0.974400      0.000904        0.974434   

   Precision_std  Recall_mean  Recall_std   F1_mean    F1_std  AUC_mean  \
0       0.000775     0.981162    0.000780  0.981151  0.000778  0.999655   
1       0.001799     0.968395    0.001817  0.968390  0.001814  0.998935   
2       0.004576     0.793114    0.004722  0.794831  0.004598  0.955147   
3       0.000903     0.974400    0.000904  0.974369 

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

In [ ]:
# Funkcja do pobrania i rozpakowania pliku gzip
def download_and_extract(url, filename, is_image=True):
    """Pobiera plik gzip i wczytuje do numpy."""
    if not os.path.exists(filename):
        print(f"Pobieranie {filename}...")
        urllib.request.urlretrieve(url, filename)
        print(f"Pobrano {filename}.")

    with gzip.open(filename, 'rb') as f:
        data = f.read()
        if is_image:
            # Pierwsze 16 bajtów to nagłówek IDX dla obrazów
            data = np.frombuffer(data, dtype=np.uint8, offset=16).astype(np.float32)
            data = data.reshape(-1, 28, 28)
        else:
            # Pierwsze 8 bajtów to nagłówek IDX dla etykiet
            data = np.frombuffer(data, dtype=np.uint8, offset=8).astype(np.float32)
    return data

# URL-e Fashion MNIST
urls = {
    "x_train": "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-images-idx3-ubyte.gz",
    "y_train": "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-labels-idx1-ubyte.gz",
    "x_test":  "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-images-idx3-ubyte.gz",
    "y_test":  "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-labels-idx1-ubyte.gz"
}

# Ścieżki lokalne
paths = {k: k + "_Fashion_Mnist.gz" for k in urls}

# Pobranie i wczytanie danych
X_train = download_and_extract(urls["x_train"], paths["x_train"], is_image=True)
y_train = download_and_extract(urls["y_train"], paths["y_train"], is_image=False)
X_test  = download_and_extract(urls["x_test"], paths["x_test"], is_image=True)
y_test  = download_and_extract(urls["y_test"], paths["y_test"], is_image=False)

X = np.concatenate((X_train, X_test))#[:100]
X = X.reshape(X.shape[0], -1)
y = np.concatenate((y_train, y_test))#[:100]

n_components = 32
LDA_n_components = min(n_components, len(np.unique(y))- 1)
print(LDA_n_components)
reducers = {    
    "PCA": lambda: PCA(n_components=n_components),
    "ICA": lambda: FastICA(
        n_components=n_components,
        random_state=765243
    ),
    "LDA": lambda: LDA(n_components= LDA_n_components),
    "UMAP": lambda: umap.UMAP(n_components=n_components),
    "ConvAE": lambda: MNISTAutoencoder(
        latent_dim=n_components,
        epochs=50,
        val_split=None
    ),
    # "VAE": lambda: MNISTVAE(
    #     latent_dim=n_components,
    #     epochs=50,
    #     val_split=None
    # )
}

scaler_getter = lambda: ImageScaler()

classifiers = {
    "SVM": lambda: SVC(kernel="rbf", probability=True, random_state=765243),
    "MLP": lambda: MLPClassifier(hidden_layer_sizes=(64,16), activation="relu", max_iter=100, random_state=765243),
    "Decision Tree": lambda: DecisionTreeClassifier(max_depth=15, random_state=765243),
    "kNN": lambda: KNeighborsClassifier(n_neighbors=10) 
}

full_pipeline(X, y, reducers, classifiers, name='Fashion_Mnist_32', print_train_plot=True, n_splits=5, n_repeats=3, scaler_getter=scaler_getter)


9
full_pipeline: PCA


c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.080852        0.000254  2.829319  0.011299   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.613203        0.101971  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.852819      0.002697        0.851411   
1    PCA            MLP       0.860219      0.002410        0.859565   
2    PCA  Decision Tree       0.783229      0.002773        0.783567   
3    PCA            kNN       0.841114      0.002105        0.840319   

   Precision_std  Recall_mean  Recall_std   F1_mean    F1_std  AUC_mean  \
0       0.002693     0.852819    0.002697  0.851366  0.002625  0.986375   
1       0.002315     0.860219    0.002410  0.859282  0.002353  0.987606   
2       0.003125     0.783229    0.002773  0.783048  0.002894  0.926735   
3       0.001915     0.841114    0.002105  0.839579 

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.080852        0.000254  2.829319  0.011299   
1    ICA         0.078758        0.000192  3.197194  0.011611   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.613203        0.101971  
1                  NaN                 NaN        12.319011        3.804546  
  Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0    PCA            SVM       0.852819      0.002697        0.851411   
1    PCA            MLP       0.860219      0.002410        0.859565   
2    PCA  Decision Tree       0.783229      0.002773        0.783567   
3    PCA            kNN       0.841114      0.002105        0.840319   
4    ICA            SVM       0.863871      0.002673        0.862765   
5    ICA            MLP       0.859667      0.003367        0.859218   
6    ICA  Decision Tree       0.739500      0.006269        0.7518

...

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.080852        0.000254  2.829319  0.011299   
1    ICA         0.078758        0.000192  3.197194  0.011611   
2    LDA         0.301191        0.000797  1.276742  0.003595   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.613203        0.101971  
1                  NaN                 NaN        12.319011        3.804546  
2                  NaN                 NaN         9.069895        1.670287  
   Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0     PCA            SVM       0.852819      0.002697        0.851411   
1     PCA            MLP       0.860219      0.002410        0.859565   
2     PCA  Decision Tree       0.783229      0.002773        0.783567   
3     PCA            kNN       0.841114      0.002105        0.840319   
4     ICA            SVM       0.863871      0.002673        0

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

  Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0    PCA         0.080852        0.000254  2.829319  0.011299   
1    ICA         0.078758        0.000192  3.197194  0.011611   
2    LDA         0.301191        0.000797  1.276742  0.003595   
3   UMAP         0.817034        0.004536  0.280734  0.007745   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.613203        0.101971  
1                  NaN                 NaN        12.319011        3.804546  
2                  NaN                 NaN         9.069895        1.670287  
3                  NaN                 NaN        70.627327       31.704750  
   Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0     PCA            SVM       0.852819      0.002697        0.851411   
1     PCA            MLP       0.860219      0.002410        0.859565   
2     PCA  Decision Tree       0.783229      0.002773        0.78

c:\Users\Cezary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


   Method  Silhouette_mean  Silhouette_std  DBI_mean   DBI_std  \
0     PCA         0.080852        0.000254  2.829319  0.011299   
1     ICA         0.078758        0.000192  3.197194  0.011611   
2     LDA         0.301191        0.000797  1.276742  0.003595   
3    UMAP         0.817034        0.004536  0.280734  0.007745   
4  ConvAE         0.084634        0.004762  2.601726  0.061004   

   Reconstruction_mean  Reconstruction_std  Train_time_mean  Train_time_std  
0                  NaN                 NaN         0.613203        0.101971  
1                  NaN                 NaN        12.319011        3.804546  
2                  NaN                 NaN         9.069895        1.670287  
3                  NaN                 NaN        70.627327       31.704750  
4             0.013494             0.00052       547.225401       67.355238  
    Method     Classifier  Accuracy_mean  Accuracy_std  Precision_mean  \
0      PCA            SVM       0.852819      0.002697       